# 04 — GPU, Mixed Precision, Profiling

Companion to [`notes.md`](notes.md).

This notebook is honestly split into two kinds of cells:

- **GPU / mixed-precision cells** (`.to("cuda")`, `torch.autocast(device_type="cuda", ...)`,
  `torch.cuda.amp.GradScaler`): written and reviewed for correctness, but **not executed** —
  this environment has no GPU. Each such cell is guarded so it prints an honest skip
  message instead of running, and is clearly labeled.
- **CPU profiling cells**: fully runnable without a GPU, and **actually executed** below,
  with real captured `torch.profiler` output.


In [1]:
import torch
import torch.nn as nn

print("torch:", torch.__version__)
print("torch.cuda.is_available():", torch.cuda.is_available())
assert torch.cuda.is_available() is False, "expected no GPU in this environment"


torch: 2.13.0+cpu
torch.cuda.is_available(): False


## GPU / mixed-precision code — written, reviewed, NOT executed here

The cell below is real, correct PyTorch code for training on a GPU with automatic mixed
precision. It is guarded by `torch.cuda.is_available()`; in this environment that guard is
`False`, so it prints a skip message and does nothing else. The code path itself has been
reviewed line-by-line against the mechanism described in `notes.md`'s "Conceptual
foundation" (autocast picks fp16/bf16 for matmul-heavy ops while keeping reduction-sensitive
ops like softmax/loss in fp32; `GradScaler` multiplies the loss before `.backward()` and
unscales gradients before `optimizer.step()`, to stop small fp16 gradients from
underflowing to zero) — but it has never actually run, on this machine or any other, as
part of this notebook.


In [2]:
def train_one_epoch_amp(model, loader, optimizer, loss_fn, device, scaler):
    """One epoch of GPU training with automatic mixed precision.

    NOT EXECUTED IN THIS ENVIRONMENT (torch.cuda.is_available() == False).
    Written and reviewed for correctness; mirrors the fp32 CPU training loop in
    03-datasets-dataloaders-checkpointing, with three GPU/AMP-specific additions marked below.
    """
    model.train()
    running_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)  # (1) move batch to GPU memory
        optimizer.zero_grad()

        # (2) autocast: ops inside this context run in fp16/bf16 where it's numerically
        # safe (matmuls, convolutions) and stay in fp32 where it isn't (loss reduction)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            pred = model(xb)
            loss = loss_fn(pred, yb)

        # (3) GradScaler: scale the loss up before backward so small fp16 gradients don't
        # underflow to zero, then unscale before the optimizer step
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * xb.size(0)
    return running_loss / len(loader.dataset)


if torch.cuda.is_available():
    # Real GPU training would happen here. Left as pseudocode-shaped setup:
    device = torch.device("cuda")
    model = nn.Sequential(nn.Linear(30, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid()).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
    loss_fn = nn.BCELoss()
    scaler = torch.cuda.amp.GradScaler()
    # train_one_epoch_amp(model, train_loader, optimizer, loss_fn, device, scaler)
    raise RuntimeError("unreachable in this environment")
else:
    print("SKIPPED: no CUDA device available in this environment.")
    print("The train_one_epoch_amp() function above is real, reviewed code -- not executed.")


SKIPPED: no CUDA device available in this environment.
The train_one_epoch_amp() function above is real, reviewed code -- not executed.


## CPU profiling — actually run

`torch.profiler` works without a GPU: pass `activities=[torch.profiler.ProfilerActivity.CPU]`
and it instruments ops running on the CPU exactly as it would instrument GPU kernels with
`ProfilerActivity.CUDA` added. Below, a real training loop — the same `MLP` architecture and
Breast Cancer dataset from `03-datasets-dataloaders-checkpointing` — is profiled for a few
epochs, and the real captured per-operator breakdown is printed.

**Hypothesis (stated before running):** for a small fully-connected MLP on a small dataset,
the matrix-multiply-adjacent ops inside `nn.Linear` (`aten::addmm` / `aten::linear`) should
dominate total CPU time, since that is where essentially all of the floating-point work in
this model happens; data loading and the Python-level loop overhead should be comparatively
small at this scale.


In [3]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.profiler import profile, ProfilerActivity

torch.manual_seed(0)

data = load_breast_cancer()
X, y = data.data, data.target.astype(np.float32)
X_train, _, y_train, _ = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
X_train = StandardScaler().fit_transform(X_train).astype(np.float32)

X_t = torch.from_numpy(X_train)
y_t = torch.from_numpy(y_train).unsqueeze(1)
train_ds = torch.utils.data.TensorDataset(X_t, y_t)
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True)


class MLP(nn.Module):
    def __init__(self, n_in, n_hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)


profile_model = MLP(n_in=X_train.shape[1], n_hidden=64)
profile_optimizer = torch.optim.Adam(profile_model.parameters(), lr=1e-2)
profile_loss_fn = nn.BCELoss()


def run_training_epochs(n_epochs):
    profile_model.train()
    for _ in range(n_epochs):
        for xb, yb in train_loader:
            profile_optimizer.zero_grad()
            pred = profile_model(xb)
            loss = profile_loss_fn(pred, yb)
            loss.backward()
            profile_optimizer.step()


# one warm-up epoch, unprofiled, so lazy one-time setup costs don't pollute the trace
run_training_epochs(1)

with profile(activities=[ProfilerActivity.CPU], record_shapes=True) as prof:
    run_training_epochs(3)

print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=12))


USDT:2026-08-24 00:33:49 183349:183349 SyncActivityProfilerHandler.cpp:52] profiler_start


USDT:2026-08-24 00:33:50 183349:183349 SyncActivityProfilerHandler.cpp:59] profiler_stop


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
    autograd::engine::evaluate_function: AddmmBackward0         0.25%       1.546ms        47.41%     297.238ms       2.202ms           135  
                                         AddmmBackward0         0.19%       1.178ms        46.90%     294.006ms       2.178ms           135  
                                               aten::mm        46.50%     291.533ms        46.51%     291.586ms       1.296ms           225  
                               Optimizer.step#Adam.step         1.88%      11.805ms        27.18%     170.368ms       3.786ms            45  
      

### Actual result

The captured trace (printed above, real output from this run) is read directly, not
summarized from memory. Recording the key numbers here for the record.


In [4]:
avgs = prof.key_averages()
sorted_avgs = sorted(avgs, key=lambda e: e.cpu_time_total, reverse=True)
top = sorted_avgs[0]
print(f"Top op by total CPU time: {top.key}  ({top.cpu_time_total / 1000:.2f} ms total, {top.count} calls)")

total_cpu_time = sum(e.cpu_time_total for e in avgs)
addmm_time = sum(e.cpu_time_total for e in avgs if 'addmm' in e.key or 'linear' in e.key or 'mm' in e.key.lower())
print(f"Share of total CPU time in matmul-family ops (addmm/linear/mm): "
      f"{100 * addmm_time / total_cpu_time:.1f}%")


Top op by total CPU time: autograd::engine::evaluate_function: AddmmBackward0  (297.24 ms total, 135 calls)
Share of total CPU time in matmul-family ops (addmm/linear/mm): 74.6%


**Interpretation:** the trace confirms the core hypothesis — matmul-family ops
(`aten::mm`/`aten::addmm`, underlying every `nn.Linear` forward pass, plus their
autograd backward-pass counterparts `AddmmBackward0`) account for the largest single share
of total CPU time (~75%), consistent with this being a model whose compute is dominated by
matrix multiplication. The trace also surfaces something the hypothesis didn't predict:
`Optimizer.step#Adam.step` (27% of total time, mostly `aten::sqrt` for Adam's per-parameter
variance normalization) is a non-trivial secondary cost — a reminder that profiling, not
intuition, is what tells you where time actually goes; a plain-SGD optimizer would not pay
this cost at all. This is exactly the kind of measurement `notes.md`'s "Failure modes"
section says *must* come before reaching for GPU acceleration or mixed precision: profiling
first identifies that the time is actually going into compute (which a GPU/AMP would help)
rather than, say, data loading (2% here) or Python-loop overhead (which they would not).

**Limitations:** this is a small model (three `nn.Linear` layers, hidden width 64) on a
small dataset (455 training examples) run for only 3 profiled epochs on one CPU — the exact
percentages will differ for a larger model, a different batch size, a different optimizer,
or a different machine's CPU, and profiler instrumentation itself adds overhead that
slightly inflates total measured time relative to an unprofiled run (see `notes.md`'s
"Failure modes").
